In [154]:
# ── Heart Failure Clinical Data ───────────────────────────────────────────────
# Source: https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data
# Task:   Predict heart failure mortality risk
# File:   heart_failure_clinical_records_dataset.csv
# ─────────────────────────────────────────────────────────────────────────────

# TODO: import any additional libraries you need
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

In [155]:
df = pd.read_csv('../data/heart_failure_clinical_records_dataset.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
df.head()

Shape: (299, 13)

Columns: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT']

Data types:
 age                         float64
anaemia                       int64
creatinine_phosphokinase      int64
diabetes                      int64
ejection_fraction             int64
high_blood_pressure           int64
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                           int64
smoking                       int64
time                          int64
DEATH_EVENT                   int64
dtype: object


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [156]:
# Quick look — run this before anything else
print("Missing values:")
print(df.isnull().sum())

print("\nTarget distribution (DEATH_EVENT):")
print(df['DEATH_EVENT'].value_counts())
print(f"Mortality rate: {df['DEATH_EVENT'].mean():.1%}")

# Things to think about:
# - Only 299 rows — how does that affect your modeling choices?
# - The 'time' column is follow-up duration. Should you use it as a feature?

Missing values:
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

Target distribution (DEATH_EVENT):
DEATH_EVENT
0    203
1     96
Name: count, dtype: int64
Mortality rate: 32.1%


In [157]:
# ── EDA ───────────────────────────────────────────────────────────────────────
# TODO: explore the data
# Ideas:
# - Plot distributions of ejection_fraction, serum_creatinine, age
# - Boxplots comparing survivors vs non-survivors across key features
# - Correlation heatmap

# your code here

In [158]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
# TODO: prepare data for modeling
# Things to consider:
# - No missing values — lucky!
# - Feature scaling (important for some models)
# - Include or exclude 'time'? Why?
# - Small dataset — consider cross-validation over a single train/test split

# your code here

# Validate Columns
req_cols = {'age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT'}
if not req_cols.issubset(df.columns):
    raise

# Handle null values (just in case)
df.fillna(0, inplace=True)

# Derive patient IDs
if 'patient_id' in df.columns:
    df.drop(columns=['patient_id'], inplace=True)
df.insert(0, 'patient_id', 'P000000')

cols = df.shape[0]
for col in range(cols):
    df.at[col, 'patient_id'] = f'P{col:06d}'

In [159]:
# ── Model ─────────────────────────────────────────────────────────────────────
# TODO: build and train your model

# your code here

# Define feature matrix + target
FEATURES = [
    'age',
    'anaemia',
    'creatinine_phosphokinase',
    'diabetes',
    'ejection_fraction',
    'high_blood_pressure',
    'platelets',
    'serum_creatinine',
    'serum_sodium',
    'sex',
    'smoking'
]

TARGET = 'DEATH_EVENT'

X = df[FEATURES]
y = df[TARGET]

# Only put scaler on training folds
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, 
                                       class_weight='balanced',
                                       max_depth=3,
                                       min_samples_leaf=15,
                                       random_state=42))
])

pipeline.fit(X, y)

with open('pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
# TODO: evaluate your model
# Suggested metrics:
# - ROC-AUC
# - Precision, Recall, F1
# - Confusion matrix

# Bonus: try building a model using ONLY serum_creatinine + ejection_fraction
# The reference paper found these two features alone predict survival well

# Discussion:
# - How does your AUC compare to random guessing (0.5)?
# - What are the risks of drawing conclusions from only 299 patients?

# your code here

# Do multiple cross-validation rounds
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
results = cross_validate(
    pipeline, X, y,
    cv=cv,
    scoring=['accuracy', 'roc_auc', 'f1'],
    return_train_score=True
)

metrics = {
    'Accuracy': results['test_accuracy'],
    'ROC-AUC': results['test_roc_auc'],
    'F1 Score': results['test_f1']
}

# Save metrics to a CSV
metrics_rows = []

for metric, scores in metrics.items():
    metrics_rows.append({
        'Metric':   metric,
        'Mean':     round(scores.mean(), 3),
        'Std':      round(scores.std(), 3),
        'Min':      round(scores.min(), 3),
        'Max':      round(scores.max(), 3)
    })

metrics_rows.append({
    'Metric': 'Overfitting Gap',
    'Mean':   round(gap, 3),
    'Std':    None,
    'Min':    None,
    'Max':    None
})

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv('metrics.csv', index=False)

print("Metrics saved.")
metrics_df

Repeated Stratified K-Fold (5 splits × 10 repeats = 50 evaluations)

Metric           Mean      Std      Min      Max
--------------------------------------------
Accuracy        0.740    0.053    0.617    0.867
ROC-AUC         0.785    0.056    0.651    0.891
F1 Score        0.611    0.082    0.343    0.778

Overfitting gap (train - test): 0.051 ⚠️ Potential overfit
Metrics saved.


,Metric,Mean,Std,Min,Max
0,Accuracy,0.740,0.053,0.617,0.867
1,ROC-AUC,0.785,0.056,0.651,0.891
2,F1 Score,0.611,0.082,0.343,0.778
3,Overfitting Gap,0.051,NaN,NaN,NaN


In [161]:
# ── Similar Patients ─────────────────────────────────────────────────────────────
# Finds patients with similar statistics to a given patient
# Also summarizes the outcomes of similar parents
def find_similar_patients(patient, df, top_n):
    continuous = [
        'age', 'creatinine_phosphokinase', 'ejection_fraction',
        'platelets', 'serum_creatinine', 'serum_sodium'
    ]
    boolean = [
        'anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking'
    ]

    # Normalize continuous features to 0-1 for fair distance comparison
    df_norm = df.copy()
    patient_norm = {}

    for col in continuous:
        col_min = df[col].min()
        col_max = df[col].max()
        df_norm[col] = (df[col] - col_min) / (col_max - col_min)
        patient_norm[col] = (patient[col] - col_min) / (col_max - col_min)

    for col in boolean:
        patient_norm[col] = patient[col]

    # Calculate Euclidean distance from patient to every row
    all_features = continuous + boolean
    distances = np.sqrt(
        sum((df_norm[col] - patient_norm[col]) ** 2 for col in all_features)
    )

    df_norm['distance'] = distances

    # Return top N most similar patients with their time + outcome
    similar = df.loc[df_norm['distance'].nsmallest(top_n).index]
    return similar[['time', 'DEATH_EVENT', 'ejection_fraction', 
                     'serum_creatinine', 'age']]

def summarize_survival(similar_patients: pd.DataFrame) -> dict:
    """
    Summarize what happened to similar patients over time.
    """
    died     = similar_patients[similar_patients['DEATH_EVENT'] == 1]
    survived = similar_patients[similar_patients['DEATH_EVENT'] == 0]

    result = {
        "similar_patients_count": len(similar_patients),
        "died_count":             len(died),
        "survived_count":         len(survived),
        "mortality_rate":         f"{len(died) / len(similar_patients) * 100:.0f}%",
    }

    if len(died) > 0:
        result["avg_days_to_death"] = round(died['time'].mean())
        result["min_days_to_death"] = int(died['time'].min())
        result["max_days_to_death"] = int(died['time'].max())
    else:
        result["avg_days_to_death"] = None

    if len(survived) > 0:
        result["avg_followup_survived"] = round(survived['time'].mean())

    return result

In [162]:
# Sort patient into risk category
def get_risk_category(risk_prob):
    if risk_prob >= 0.7:
        return {
            "category": "CRITICAL",
            "label": "Critical Risk",
            "urgency": "critical"
        }
    elif risk_prob >= 0.4:
        return {
            "category": "MODERATE",
            "label": "Moderate Risk",
            "urgency": "moderate"
        }
    else:
        return {
            "category": "LOW",
            "label": "Low Risk",
            "urgency": "low"
        }

# Generate a list of risk factors depending on patient metrics
def get_risk_factors(patient: dict) -> list:
    factors = []

    if patient['ejection_fraction'] < 40:
        factors.append(
            f"Low ejection fraction ({patient['ejection_fraction']}%) — "
            f"heart is pumping poorly"
        )

    if patient['serum_creatinine'] > 2.0:
        factors.append(
            f"High serum creatinine ({patient['serum_creatinine']} mg/dL) — "
            f"indicates kidney stress"
        )

    if patient['age'] > 70:
        factors.append(
            f"Advanced age ({patient['age']}) — "
            f"significantly increases cardiac risk"
        )

    if patient['serum_sodium'] < 130:
        factors.append(
            f"Low serum sodium ({patient['serum_sodium']} mEq/L) — "
            f"suggests advanced fluid retention"
        )

    if patient['diabetes'] == 1:
        factors.append(
            "Diabetic — compounds cardiovascular risk"
        )

    if patient['high_blood_pressure'] == 1:
        factors.append(
            "High blood pressure — increases strain on the heart"
        )

    if patient['anaemia'] == 1:
        factors.append(
            "Anaemia — reduced oxygen delivery to the heart"
        )

    if patient['creatinine_phosphokinase'] > 1000:
        factors.append(
            f"Elevated CPK ({patient['creatinine_phosphokinase']} U/L) — "
            f"indicates muscle or cardiac stress"
        )

    if patient['smoking'] == 1:
        factors.append(
            "Smoker — accelerates arterial damage"
        )

    if patient['platelets'] < 100:
        factors.append(
            f"Low platelets ({patient['platelets']}k/µL) — "
            f"suggests systemic illness"
        )
    elif patient['platelets'] > 600:
        factors.append(
            f"High platelets ({patient['platelets']}k/µL) — "
            f"elevated clotting risk"
        )

    return factors if factors else ["No individual risk factors flagged"]

# Recommend a course of action based on their category
def get_recommendation(risk_category: dict, survival: dict,
                        risk_factors: list) -> dict:
    """
    Map risk category + survival stats + risk factors
    to a full recommendation object for the UI.
    """
    category  = risk_category["category"]
    avg_days  = survival.get("avg_days_to_death")
    mort_rate = survival["mortality_rate"]
    factors_text = "; ".join(risk_factors)

    if category == "CRITICAL":
        title   = "⛔ Admit Immediately"
        summary = (
            f"This patient is at critical risk of mortality. "
            f"{mort_rate} of similar patients died"
            + (f", on average within {avg_days} days." if avg_days else ".")
        )
        action  = (
            "Immediate admission and intervention is strongly recommended. "
            "Do not discharge without cardiology review."
        )

    elif category == "MODERATE":
        title   = "⚠️ Monitor Closely"
        summary = (
            f"This patient shows moderate mortality risk. "
            f"{mort_rate} of similar patients died"
            + (f", typically within {avg_days} days." if avg_days else ".")
        )
        action  = (
            "Schedule follow-up within 7 days. Monitor serum creatinine "
            "and ejection fraction closely. Consider cardiology referral."
        )

    else:
        survived = survival.get("avg_followup_survived", "N/A")
        title    = "✅ Routine Care"
        summary  = (
            f"This patient is currently low risk. "
            f"{survival['survived_count']} of {survival['similar_patients_count']} "
            f"similar patients survived with an average follow-up of "
            f"{survived} days."
        )
        action   = (
            "Continue routine monitoring. Reassess if symptoms worsen "
            "or new risk factors emerge."
        )

    return {
        "title":        title,
        "summary":      summary,
        "action":       action,
        "risk_factors": risk_factors,
        "factors_text": factors_text
    }


def predict_patient(patient: dict) -> dict:
    # Step 1 — Risk probability
    df_input  = pd.DataFrame([patient])[FEATURES]
    df_scaled = scaler.transform(df_input)
    risk_prob = model.predict_proba(df_scaled)[0][1]

    # Step 2 — Risk category
    risk_category = get_risk_category(risk_prob)

    # Step 3 — Identify contributing risk factors
    risk_factors = get_risk_factors(patient)

    # Step 4 — Similar patient survival lookup
    similar  = find_similar_patients(patient, df)
    survival = summarize_survival(similar)

    # Step 5 — Map everything to recommendation
    recommendation = get_recommendation(risk_category, survival, risk_factors)

    return {
        "mortality_risk":   f"{risk_prob * 100:.1f}%",
        "risk_category":    risk_category,
        "similar_patients": survival,
        "recommendation":   recommendation
    }